# Student trajectory characterization: Qwen3.5-2B

256 prompts from OpenThoughts, one trajectory each, sampled with the recommended
Qwen3.5 thinking-mode settings (`temperature=1.0`, `top_p=0.95`, `top_k=20`,
`presence_penalty=1.5`), seed 42, split across two A100s.

**No generation cap.** Traces run until the model emits EOS, with a 40,000-token
degeneracy guard as a backstop. This matters: an earlier pass with a 4,096-token cap
reported a mean trace of 3,929 tokens, which was purely an artifact of the cap.

Two questions this notebook answers:
1. How long are the trajectories?
2. How many are correct, graded with `math_verify` against the dataset's gold answer?

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

GUARD = 40_000  # degeneracy backstop passed as --max-new-tokens

rows = []
for shard in (0, 1):
    with open(f'outputs/student256/trajectories.shard{shard}.jsonl') as fh:
        rows += [json.loads(line) for line in fh]

df = pd.DataFrame(rows)
print(f'{len(df)} trajectories, {df.response_length.sum():,} generated tokens')
df[['id', 'source', 'response_length', 'correct', 'truncated', 'has_boxed']].head()

## 1. Token lengths

These are much longer than a 2B model on math might suggest.

In [ ]:
L = df.response_length
pd.Series({
    'mean': L.mean(), 'median': L.median(), 'min': L.min(), 'max': L.max(),
    **{f'p{p}': L.quantile(p/100) for p in (5, 25, 50, 75, 90, 95)},
}).astype(int).to_frame('tokens')

Everything from p75 upward sits at exactly 40,000. That is the guard, not a property
of the model — the distribution is bimodal and the upper mode is a wall.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(L, bins=40, color='#4C72B0', edgecolor='white')
ax.axvline(GUARD, color='crimson', ls='--', lw=2, label=f'{GUARD:,}-token guard')
ax.axvline(L.median(), color='#DD8452', ls='-', lw=2, label=f'median {L.median():,.0f}')
ax.set_xlabel('response length (tokens)')
ax.set_ylabel('trajectories')
ax.set_title('Trace length is bimodal: a broad distribution plus a spike at the guard')
ax.legend()
plt.tight_layout()

In [ ]:
bins   = [0, 2_000, 5_000, 10_000, 20_000, 30_000, GUARD - 1, np.inf]
labels = ['<2k', '2k-5k', '5k-10k', '10k-20k', '20k-30k', '30k-40k', 'hit guard']
df['bucket'] = pd.cut(df.response_length, bins=bins, labels=labels, right=False)

by_len = df.groupby('bucket', observed=True).agg(
    n=('correct', 'size'), correct=('correct', 'sum'))
by_len['share_%']   = (by_len.n / len(df) * 100).round(1)
by_len['correct_%'] = (by_len.correct / by_len.n * 100).round(1)
by_len

## 2. Correctness

`correct` is `math_verify` comparing the model's extracted answer against the
dataset's gold `Answer` field. Note the grading ceiling: `math_verify` scores the
dataset's *own* reference solutions against its *own* golds at only ~97-98%, so that
is the effective maximum, not 100%.

In [ ]:
n_correct = int(df.correct.sum())
print(f'overall: {n_correct}/{len(df)} = {n_correct/len(df):.1%} correct')
print(f'wrong:   {len(df)-n_correct}/{len(df)} = {1-n_correct/len(df):.1%}')

But that single number blends two populations that behave completely differently.

In [ ]:
by_end = df.groupby('truncated').agg(
    n=('correct', 'size'), mean_len=('response_length', 'mean'),
    correct=('correct', 'sum'))
by_end.index = by_end.index.map({False: 'finished on EOS', True: 'hit 40k guard'})
by_end['correct_%'] = (by_end.correct / by_end.n * 100).round(1)
by_end['mean_len']  = by_end.mean_len.astype(int)
by_end

**When the model reaches its own conclusion it is ~88% correct** — close to the
97-98% grading ceiling. When it runs into the guard it is ~27%.

So the dominant failure mode is *not finishing*, not faulty reasoning.

## 3. A caveat that changes the headline number

`math_verify` extracts an answer even when the trace never wrote a `\boxed{}`.
For a trace severed mid-reasoning, whatever it picks up is scratch work.

In [ ]:
by_box = df.groupby('has_boxed').agg(
    n=('correct', 'size'), truncated=('truncated', 'sum'), correct=('correct', 'sum'))
by_box.index = by_box.index.map({False: 'no \\boxed{}', True: 'has \\boxed{}'})
by_box['correct_%'] = (by_box.correct / by_box.n * 100).round(1)
by_box

Every trace without a `\boxed{}` is a truncated one. Those never produced an answer
at all, so their 'correct' scores come from fallback extraction and are unreliable.

This gives an honest range rather than a single figure.

In [ ]:
boxed = df[df.has_boxed]
optimistic  = df.correct.sum() / len(df)
conservative = boxed.correct.sum() / len(df)   # no answer == failure
answered     = boxed.correct.sum() / len(boxed)

pd.Series({
    'optimistic (credits fallback extraction)': optimistic,
    'conservative (no answer = failure)':       conservative,
    'accuracy when it did answer':             answered,
}).map('{:.1%}'.format).to_frame('correct')

## 4. By source

The sample is dominated by its hardest source, which drives both length and errors.

In [ ]:
by_src = df.groupby('source').agg(
    n=('correct', 'size'), correct=('correct', 'sum'),
    median_len=('response_length', 'median'), truncated=('truncated', 'sum'))
by_src['correct_%']   = (by_src.correct / by_src.n * 100).round(1)
by_src['truncated_%'] = (by_src.truncated / by_src.n * 100).round(1)
by_src.sort_values('n', ascending=False)

## 5. Conclusions

1. **Traces are long**: median ~30.7k tokens, mean ~26.9k. Any cap below ~40k
   materially distorts the data; the old 4,096 cap distorted it beyond use.
2. **The 40k guard is now the main measurement artifact**, binding on ~39% of traces.
   It was intended as a rare degeneracy backstop and is instead shaping two-fifths
   of the run.
3. **Reasoning quality is good; termination is the problem.** ~88% correct on traces
   that finish, against a ~97-98% grading ceiling.
4. **Report the range, not one number**: ~55% conservative to ~64% optimistic overall,
   and ~87% on the traces that actually produced an answer.

Open question for the next pass: `presence_penalty=1.5` permanently down-weights every
token already seen. Over a 30k-token trace that pushes the model away from ordinary
vocabulary exactly when it should be concluding, so it is a plausible contributor to
the non-termination — not merely a victim of it. A short A/B at 0.0 vs 1.5 on the
prompts that hit the guard would settle it.